# Construire et ingérer la couche Bronze

## Objectif concret

Ce notebook décrit et vérifie le socle réellement mis en place pour stocker les trois CSV Indusense dans PostgreSQL : un modèle SQLAlchemy par table Bronze, une migration Alembic versionnée et une ingestion idempotente. À la fin, on sait expliquer ce qui est conservé tel quel, comment le schéma est créé et pourquoi un rejeu n'ajoute pas de doublons.

**Point de départ :** exécuter ce notebook depuis le dossier racine `indusense`, après `uv sync`. Les cellules de lecture ne modifient aucun fichier ni aucune table.

In [1]:
from pathlib import Path

# Le notebook se trouve à la racine du projet : ce chemin rend les accès aux CSV explicites.
PROJECT_ROOT = Path.cwd()
DATA_DIRECTORY = PROJECT_ROOT / 'datas'
assert (PROJECT_ROOT / 'pyproject.toml').exists(), (
    'Ouvrir le notebook depuis le dossier racine indusense.'
)

print(f'Projet : {PROJECT_ROOT.name}')
print(f'Dossier des sources : {DATA_DIRECTORY}')

Projet : indusense
Dossier des sources : D:\source\A4U\FormationIA\indusense\datas


## 1. Fixer le contrat Bronze

La couche **Bronze** est la zone d'atterrissage fidèle aux sources. Ici, la règle est **un fichier CSV = une table PostgreSQL**. Les colonnes métier restent en `Text`, y compris lorsqu'une valeur ressemble à une date ou un nombre : les conversions et la normalisation appartiennent à Silver.

`machine.csv` garde donc son grain mixte machine-maintenance. Le découper maintenant en tables `machine` et `maintenance` changerait déjà le sens de la source.

In [2]:
import csv

SOURCES = {
    'machine': DATA_DIRECTORY / 'machine.csv',
    'incident': DATA_DIRECTORY / 'releves_incidents.csv.csv',
    'telemetry': DATA_DIRECTORY / 'telemetry.csv.csv',
}

for name, path in SOURCES.items():
    with path.open(encoding='utf-8', newline='') as source_file:
        reader = csv.reader(source_file)
        header = next(reader)
        row_count = sum(1 for _ in reader)
    print(f'{name:9} -> {path.name:28} | {row_count:>6} lignes | {len(header):>2} colonnes')

machine   -> machine.csv                  |    115 lignes | 17 colonnes
incident  -> releves_incidents.csv.csv    |   1245 lignes | 18 colonnes
telemetry -> telemetry.csv.csv            | 135626 lignes |  7 colonnes


## 2. Relier les classes Python aux tables PostgreSQL

Un **ORM** (*Object-Relational Mapper*) associe une classe Python à une table relationnelle. La convention retenue facilite la relecture : un module de modèle par table.

Chaque table Bronze reçoit aussi des colonnes techniques : `id` est une clé interne, `batch_id` rattache la ligne à son import, `source_row_number` conserve le numéro de ligne du CSV, `record_hash` identifie le contenu brut et `ingested_at` date l'écriture. La contrainte unique porte sur `(batch_id, source_row_number)`, pas sur le hash : deux lignes identiques réellement présentes dans une source restent donc conservées.

In [3]:
from indusense.db.models import IncidentRaw, IngestionBatch, MachineMaintenanceRaw, TelemetryRaw

MODELS = [MachineMaintenanceRaw, IncidentRaw, TelemetryRaw, IngestionBatch]

for model in MODELS:
    table = model.__table__
    qualified_name = f'{table.schema}.{table.name}'
    print(f'{qualified_name:32} | {len(table.columns):>2} colonnes')

bronze_columns = set(MachineMaintenanceRaw.__table__.columns.keys())
assert {'batch_id', 'source_row_number', 'record_hash', 'ingested_at'} <= bronze_columns
assert IngestionBatch.__table__.schema == 'ops'
print('Contrat ORM vérifié : 3 tables Bronze et 1 table Ops.')

bronze.machine_maintenance_raw   | 22 colonnes
bronze.incident_raw              | 23 colonnes
bronze.telemetry_raw             | 12 colonnes
ops.ingestion_batch              |  8 colonnes
Contrat ORM vérifié : 3 tables Bronze et 1 table Ops.


## 3. Versionner le schéma avec Alembic

Une **migration** est un fichier versionné qui décrit une évolution de structure de base : schémas, tables, colonnes et contraintes. Alembic applique les migrations dans l'ordre et mémorise la révision appliquée.

La première révision crée les schémas `bronze` et `ops`, puis les quatre tables. La commande ci-dessous est à lancer dans un terminal PowerShell ouvert dans `indusense`, avec les variables `DB_*` de l'environnement déjà définies. `upgrade head` signifie « appliquer toutes les migrations jusqu'à la dernière version connue ».

In [4]:
MIGRATION = PROJECT_ROOT / 'migrations' / 'versions' / '20260831_01_create_bronze_and_ops_schema.py'
migration_text = MIGRATION.read_text(encoding='utf-8')

assert '20260831_01' in migration_text
assert 'CREATE SCHEMA IF NOT EXISTS bronze' in migration_text
assert 'CREATE SCHEMA IF NOT EXISTS ops' in migration_text

print(f'Migration trouvée : {MIGRATION.name}')
print('À exécuter une seule fois par base cible : uv run alembic upgrade head')
print('À contrôler ensuite : uv run alembic current')

Migration trouvée : 20260831_01_create_bronze_and_ops_schema.py
À exécuter une seule fois par base cible : uv run alembic upgrade head
À contrôler ensuite : uv run alembic current


## 4. Ingestion transactionnelle et idempotente

L'ingestion lit chaque CSV avec `csv.DictReader`, préserve les cellules vides comme chaînes vides, calcule une empreinte **SHA-256** du fichier et une empreinte de chaque ligne. Un **lot** est enregistré dans `ops.ingestion_batch` avec son statut. Les inserts d'une source et le passage du lot à `completed` sont validés ensemble dans une transaction.

Une ingestion est **idempotente** si on peut la rejouer sans modifier le résultat déjà correct. Ici, si un lot `completed` porte le même nom de fichier et la même empreinte SHA-256, la source est ignorée. Une erreur marque le lot `failed` et la transaction d'écriture Bronze est annulée.

In [5]:
from indusense.ingestion.bronze import read_source_rows, source_record_hash, source_sha256

for name, path in SOURCES.items():
    first_row_number, first_row = next(read_source_rows(path))
    file_hash = source_sha256(path)
    row_hash = source_record_hash(first_row)
    assert first_row_number == 2  # La ligne 1 est l'en-tête CSV.
    assert len(file_hash) == len(row_hash) == 64
    print(f'{name:9} | ligne source {first_row_number} | SHA-256 fichier {file_hash[:12]}...')

print('Les empreintes sont calculées sans modifier les fichiers CSV.')

machine   | ligne source 2 | SHA-256 fichier bf945e954d26...
incident  | ligne source 2 | SHA-256 fichier be7204b3320a...
telemetry | ligne source 2 | SHA-256 fichier 68ec8dd09b69...
Les empreintes sont calculées sans modifier les fichiers CSV.


## 5. Lancer puis vérifier l'ingestion

Dans un terminal PowerShell, depuis `indusense`, lancer d'abord `uv run indusense db-check`. Cette commande vérifie la connexion sans écrire. Ensuite, `uv run indusense ingest-bronze --source all` charge les trois sources. `--source machine`, `incident` ou `telemetry` permet de cibler une seule source.

Le résultat attendu d'une première ingestion est : 115 lignes machine-maintenance, 1 245 incidents et 135 626 mesures de télémétrie. Un second lancement affiche que chaque source est ignorée : c'est la preuve opérationnelle de l'idempotence. La cellule suivante est désactivée par défaut pour qu'un « Run All » reste en lecture seule.

In [6]:
VALIDER_POSTGRESQL = False

if VALIDER_POSTGRESQL:
    from sqlalchemy import text
    from indusense.db.engine import create_database_engine

    expected_counts = {
        'bronze.machine_maintenance_raw': 115,
        'bronze.incident_raw': 1_245,
        'bronze.telemetry_raw': 135_626,
    }
    engine = create_database_engine()
    try:
        with engine.connect() as connection:
            for table_name, expected_count in expected_counts.items():
                observed_count = connection.scalar(text(f'SELECT COUNT(*) FROM {table_name}'))
                print(f'{table_name}: {observed_count} lignes')
                assert observed_count == expected_count
    finally:
        engine.dispose()
    print('Validation PostgreSQL réussie : les volumes attendus sont présents.')
else:
    print('Contrôle PostgreSQL non exécuté. Passer VALIDER_POSTGRESQL à True après configuration DB_*.')

Contrôle PostgreSQL non exécuté. Passer VALIDER_POSTGRESQL à True après configuration DB_*.


## Synthèse à savoir expliquer

- **Fidélité Bronze :** une table par fichier, valeurs source en texte, pas de normalisation anticipée.
- **Traçabilité :** les métadonnées techniques permettent de retrouver le lot, la ligne source et l'empreinte.
- **Migration :** Alembic crée une structure versionnée et reproductible ; `create_all()` ne remplace pas cet historique.
- **Transaction :** une source est écrite de façon cohérente ou annulée en cas d'erreur.
- **Idempotence :** rejouer un fichier déjà terminé et inchangé ne crée pas de doublon.

Point restant à pratiquer : provoquer volontairement une erreur d'insertion dans un environnement isolé, puis vérifier qu'aucune ligne partielle ne reste dans Bronze.